In [1]:
# 04_calibration.ipynb
# Probability calibration (Platt scaling) for the sex-stratified XGBoost models.
# Uses leak-free out-of-fold (5-fold CV) predicted probabilities to report Brier
# score and Expected Calibration Error (ECE) before and after calibration, saves
# calibrated risk scores for the full sample (for downstream lift/DCA/counterfactual
# stages), refits a deployment XGBoost per sex on all data, and plots grayscale
# calibration curves (no titles/captions; dpi 600; PNG+PDF).

import warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
SEED = 42; np.random.seed(SEED)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"; TAB = ROOT / "results" / "tables"
FIG = ROOT / "results" / "figures"; MODELS = ROOT / "models"

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 120, "pdf.fonttype": 42,
                     "ps.fonttype": 42, "font.size": 11, "axes.edgecolor": "0.2",
                     "axes.linewidth": 0.8, "grid.color": "0.85"})
def savefig(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight"); plt.close(fig)

PRED = ["age", "HE_BMI", "HE_wc", "HE_wt", "HE_ht", "N_EN", "N_CHO", "N_SUGAR",
        "N_CHOL", "sm_presnt", "dr_month", "pa_aerobic", "L_BR_FQ", "edu", "ho_incm", "D_1_1"]

def ece(y, p, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1); e = 0.0
    for i in range(n_bins):
        m = (p > bins[i]) & (p <= bins[i + 1])
        if m.sum() > 0:
            e += (m.mean()) * abs(y[m].mean() - p[m].mean())
    return e

rows = []
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
for ax, sx in zip(axes, ["Male", "Female"]):
    s = pd.read_parquet(DATA / f"split_{sx.lower()}.parquet")
    X = s[PRED].values; y = s["diabetes"].values
    params = json.load(open(MODELS / f"xgb_{sx.lower()}_params.json"))
    base = xgb.XGBClassifier(**params, eval_metric="logloss", random_state=SEED, n_jobs=4)
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    # leak-free out-of-fold raw probabilities
    p_raw = cross_val_predict(base, X, y, cv=cv, method="predict_proba", n_jobs=4)[:, 1]
    # Platt scaling: logistic regression on the OOF logit
    eps = 1e-6; logit = np.log(np.clip(p_raw, eps, 1 - eps) / np.clip(1 - p_raw, eps, 1 - eps))
    platt = LogisticRegression().fit(logit.reshape(-1, 1), y)
    p_cal = platt.predict_proba(logit.reshape(-1, 1))[:, 1]
    rows.append([sx, "Pre-calibration", round(brier_score_loss(y, p_raw), 3), round(ece(y, p_raw), 3)])
    rows.append([sx, "Post-calibration", round(brier_score_loss(y, p_cal), 3), round(ece(y, p_cal), 3)])
    # deployment model on all data + save Platt params
    deploy = xgb.XGBClassifier(**params, eval_metric="logloss", random_state=SEED, n_jobs=4).fit(X, y)
    deploy.save_model(MODELS / f"xgb_{sx.lower()}_deploy.json")
    json.dump({"coef": float(platt.coef_[0][0]), "intercept": float(platt.intercept_[0])},
              open(MODELS / f"platt_{sx.lower()}.json", "w"))
    out = s.copy(); out["prob_raw"] = p_raw; out["prob_cal"] = p_cal
    out.to_parquet(DATA / f"scored_{sx.lower()}.parquet", index=False)
    # calibration curves
    for p, lab, c, mk in [(p_raw, "Uncalibrated", "#a9a9a9", "o"), (p_cal, "Platt-scaled", "#3b3b3b", "s")]:
        frac, mean = calibration_curve(y, p, n_bins=10, strategy="quantile")
        ax.plot(mean, frac, marker=mk, color=c, label=lab, linewidth=1.3, markersize=5)
    ax.plot([0, 1], [0, 1], ls=":", color="0.4", lw=1)
    ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Observed frequency")
    ax.text(0.04, 0.92, "(A) Male" if sx == "Male" else "(B) Female",
            transform=ax.transAxes, fontsize=10, fontweight="bold")
    ax.legend(frameon=False, loc="lower right")
savefig(fig, "fig_calibration_male_female")

cal = pd.DataFrame(rows, columns=["Sex", "Stage", "Brier", "ECE"])
cal.to_csv(TAB / "table_calibration.csv", index=False)
print("Calibration (leak-free out-of-fold):")
print(cal.to_string(index=False))
print("\nSaved scored_{male,female}.parquet, deployment models, Platt params, calibration figure.")


Calibration (leak-free out-of-fold):
   Sex            Stage  Brier   ECE
  Male  Pre-calibration  0.132 0.010
  Male Post-calibration  0.132 0.010
Female  Pre-calibration  0.100 0.006
Female Post-calibration  0.100 0.007

Saved scored_{male,female}.parquet, deployment models, Platt params, calibration figure.
